# 30. Supervisor Package

This notebook creates a compact supervisor review package for the controlled 50-painting thesis evaluation.

The package summarizes:

- what was originally proposed,
- what has been implemented so far,
- how the current results answer the research questions,
- where the implementation deviates from the proposal,
- what methodological clarifications are needed from the supervisor,
- what the next thesis steps should be.

The notebook does not run new restoration experiments. It packages and interprets existing results.

## Supervisor package purpose

The project now contains many generated artifacts: notebooks, reports, CSV summaries, visual figures, metadata files, dashboard assets, and documentation notes.

The supervisor package is intended to avoid sending the supervisor a raw repository dump.

Instead, it creates a focused review bundle containing:

- short project status,
- proposal-to-current-work alignment,
- research-question coverage,
- key results,
- methodological deviations and justifications,
- open supervisor questions,
- recommended next steps,
- selected compact summaries,
- selected visual examples,
- links or copies of the main reports.

The supervisor package should make the current thesis state understandable without requiring the supervisor to open every notebook.

## Proposal alignment focus

The remembered proposal framing includes three main research directions:

1. **Trustworthy multi-metric evaluation**
   - Whether restoration trustworthiness can be assessed better using multiple metric families than using PSNR/SSIM alone.

2. **Pretrained model comparison across painting conditions**
   - Whether pretrained restoration/inpainting models behave differently across painting categories and synthetic damage types.

3. **Uncertainty from multiple restoration candidates**
   - Whether multiple diffusion outputs for the same damaged input reveal uncertainty or instability that should affect trust in the restoration.

This supervisor package explicitly maps the completed experiment back to these research directions.

In [1]:
from pathlib import Path
import sys
import json
import shutil
from datetime import datetime

import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("Project root:", PROJECT_ROOT)
print("Source path:", src_path)

Project root: D:\Masters\FH\Thesis\painting-restoration-eval
Source path: D:\Masters\FH\Thesis\painting-restoration-eval\src


In [2]:
config_path = PROJECT_ROOT / "config" / "experiment_50_config.yaml"

if not config_path.exists():
    raise FileNotFoundError(f"Config file not found: {config_path}")

with open(config_path, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

paths_cfg = config["paths"]

processed_metadata_dir = PROJECT_ROOT / paths_cfg["processed_metadata_dir"]
metrics_dir = PROJECT_ROOT / paths_cfg["metrics_dir"]
figures_dir = PROJECT_ROOT / paths_cfg["figures_dir"]
reports_dir = PROJECT_ROOT / paths_cfg.get("reports_dir", "outputs/reports")
dashboard_dir = PROJECT_ROOT / "outputs" / "dashboard"

supervisor_package_dir = PROJECT_ROOT / "outputs" / "supervisor_package"
supervisor_data_dir = supervisor_package_dir / "data"
supervisor_figures_dir = supervisor_package_dir / "selected_figures"
supervisor_reports_dir = supervisor_package_dir / "reports"
supervisor_docs_dir = supervisor_package_dir / "docs"

for directory in [
    supervisor_package_dir,
    supervisor_data_dir,
    supervisor_figures_dir,
    supervisor_reports_dir,
    supervisor_docs_dir,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Metrics dir:", metrics_dir)
print("Figures dir:", figures_dir)
print("Reports dir:", reports_dir)
print("Dashboard dir:", dashboard_dir)
print("Supervisor package dir:", supervisor_package_dir)

Metrics dir: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics
Figures dir: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\figures
Reports dir: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports
Dashboard dir: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard
Supervisor package dir: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package


In [3]:
source_paths = {
    # Final controlled report outputs
    "final_key_results": metrics_dir / "final_controlled_50_key_results_summary.csv",
    "final_dataset_summary": metrics_dir / "final_controlled_50_dataset_summary.csv",
    "final_model_stack": metrics_dir / "final_controlled_50_model_stack_summary.csv",
    "final_metric_policy": metrics_dir / "final_controlled_50_metric_policy_summary.csv",
    "final_model_win_summary": metrics_dir / "final_controlled_50_model_win_summary.csv",
    "final_per_metric_winner_summary": metrics_dir / "final_controlled_50_per_metric_winner_summary.csv",
    "final_uncertainty_summary": metrics_dir / "final_controlled_50_uncertainty_summary.csv",
    "final_sdxl_feasibility": metrics_dir / "final_controlled_50_sdxl_feasibility_summary.csv",
    "final_visual_case_summary": metrics_dir / "final_controlled_50_visual_case_summary.csv",
    "final_visual_cases": metrics_dir / "final_controlled_50_visual_cases.csv",

    # Refined model comparison
    "refined_comparison": metrics_dir / "comparison_unified_refined_opencv_lama_stable_diffusion_50.csv",
    "refined_summary_by_mask_type": metrics_dir / "comparison_summary_by_mask_type_refined_opencv_lama_stable_diffusion_50.csv",
    "refined_summary_by_category": metrics_dir / "comparison_summary_by_category_refined_opencv_lama_stable_diffusion_50.csv",
    "refined_metric_disagreement": metrics_dir / "comparison_metric_disagreement_cases_refined_opencv_lama_stable_diffusion_50.csv",

    # Stable Diffusion uncertainty
    "uncertainty_combined_by_case": metrics_dir / "stable_diffusion_uncertainty_combined_summary_by_case_50.csv",
    "uncertainty_by_mask_type": metrics_dir / "stable_diffusion_uncertainty_combined_summary_by_mask_type_50.csv",
    "uncertainty_by_category": metrics_dir / "stable_diffusion_uncertainty_combined_summary_by_category_50.csv",
    "uncertainty_vs_refined": metrics_dir / "stable_diffusion_uncertainty_vs_refined_performance_50.csv",
    "uncertainty_quadrants": metrics_dir / "stable_diffusion_uncertainty_performance_quadrants_50.csv",

    # Dashboard manifests
    "dashboard_overview": dashboard_dir / "manifests" / "dashboard_overview_summary.json",
    "dashboard_assets_manifest": dashboard_dir / "manifests" / "dashboard_assets_manifest.json",
    "dashboard_key_findings": dashboard_dir / "manifests" / "dashboard_key_findings.json",
    "dashboard_reports_manifest": dashboard_dir / "manifests" / "dashboard_reports_manifest.json",

    # Main reports
    "final_report": reports_dir / "final_controlled_50_evaluation_report.html",
    "refined_comparison_report": reports_dir / "opencv_lama_stable_diffusion_refined_metric_comparison_report_50.html",
    "uncertainty_report": reports_dir / "stable_diffusion_uncertainty_report_50.html",

    # Project docs
    "methodology_notes": PROJECT_ROOT / "docs" / "methodology_notes.md",
    "literature_reference_log": PROJECT_ROOT / "docs" / "literature_reference_log.md",
    "model_audit_notes": PROJECT_ROOT / "docs" / "model_audit_notes.md",
}

for name, path in source_paths.items():
    print(f"{name}: {path}")

final_key_results: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\final_controlled_50_key_results_summary.csv
final_dataset_summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\final_controlled_50_dataset_summary.csv
final_model_stack: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\final_controlled_50_model_stack_summary.csv
final_metric_policy: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\final_controlled_50_metric_policy_summary.csv
final_model_win_summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\final_controlled_50_model_win_summary.csv
final_per_metric_winner_summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\final_controlled_50_per_metric_winner_summary.csv
final_uncertainty_summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\final_controlled_50_uncertainty_summary.csv
final_sdxl_feasibility: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\met

In [4]:
required_source_names = [
    "final_key_results",
    "final_dataset_summary",
    "final_model_stack",
    "final_metric_policy",
    "final_model_win_summary",
    "final_per_metric_winner_summary",
    "final_uncertainty_summary",
    "final_sdxl_feasibility",
    "final_visual_case_summary",
    "final_visual_cases",
    "refined_comparison",
    "refined_summary_by_mask_type",
    "refined_summary_by_category",
    "refined_metric_disagreement",
    "uncertainty_combined_by_case",
    "uncertainty_by_mask_type",
    "uncertainty_by_category",
    "uncertainty_vs_refined",
    "uncertainty_quadrants",
    "dashboard_overview",
    "dashboard_assets_manifest",
    "dashboard_key_findings",
    "dashboard_reports_manifest",
    "final_report",
    "refined_comparison_report",
    "uncertainty_report",
    "methodology_notes",
    "literature_reference_log",
    "model_audit_notes",
]

missing_sources = [
    (name, source_paths[name])
    for name in required_source_names
    if not source_paths[name].exists()
]

if missing_sources:
    for name, path in missing_sources:
        print(f"Missing source {name}: {path}")
    raise FileNotFoundError("Some required supervisor-package source files are missing.")

print("All required supervisor-package source files exist.")

All required supervisor-package source files exist.


In [5]:
final_key_results_df = pd.read_csv(source_paths["final_key_results"])
final_dataset_summary_df = pd.read_csv(source_paths["final_dataset_summary"])
final_model_stack_df = pd.read_csv(source_paths["final_model_stack"])
final_metric_policy_df = pd.read_csv(source_paths["final_metric_policy"])
final_model_win_summary_df = pd.read_csv(source_paths["final_model_win_summary"])
final_per_metric_winner_summary_df = pd.read_csv(source_paths["final_per_metric_winner_summary"])
final_uncertainty_summary_df = pd.read_csv(source_paths["final_uncertainty_summary"])
final_sdxl_feasibility_df = pd.read_csv(source_paths["final_sdxl_feasibility"])
final_visual_case_summary_df = pd.read_csv(source_paths["final_visual_case_summary"])
final_visual_cases_df = pd.read_csv(source_paths["final_visual_cases"])

refined_comparison_df = pd.read_csv(source_paths["refined_comparison"])
refined_summary_by_mask_type_df = pd.read_csv(source_paths["refined_summary_by_mask_type"])
refined_summary_by_category_df = pd.read_csv(source_paths["refined_summary_by_category"])
refined_metric_disagreement_df = pd.read_csv(source_paths["refined_metric_disagreement"])

uncertainty_combined_by_case_df = pd.read_csv(source_paths["uncertainty_combined_by_case"])
uncertainty_by_mask_type_df = pd.read_csv(source_paths["uncertainty_by_mask_type"])
uncertainty_by_category_df = pd.read_csv(source_paths["uncertainty_by_category"])
uncertainty_vs_refined_df = pd.read_csv(source_paths["uncertainty_vs_refined"])
uncertainty_quadrants_df = pd.read_csv(source_paths["uncertainty_quadrants"])

dashboard_overview = json.loads(source_paths["dashboard_overview"].read_text(encoding="utf-8"))
dashboard_assets_manifest = json.loads(source_paths["dashboard_assets_manifest"].read_text(encoding="utf-8"))
dashboard_key_findings = json.loads(source_paths["dashboard_key_findings"].read_text(encoding="utf-8"))
dashboard_reports_manifest = json.loads(source_paths["dashboard_reports_manifest"].read_text(encoding="utf-8"))

loaded_tables = {
    "final_key_results": final_key_results_df,
    "final_dataset_summary": final_dataset_summary_df,
    "final_model_stack": final_model_stack_df,
    "final_metric_policy": final_metric_policy_df,
    "final_model_win_summary": final_model_win_summary_df,
    "final_uncertainty_summary": final_uncertainty_summary_df,
    "final_sdxl_feasibility": final_sdxl_feasibility_df,
    "final_visual_cases": final_visual_cases_df,
    "refined_comparison": refined_comparison_df,
    "refined_metric_disagreement": refined_metric_disagreement_df,
    "uncertainty_combined_by_case": uncertainty_combined_by_case_df,
    "uncertainty_vs_refined": uncertainty_vs_refined_df,
}

for name, df in loaded_tables.items():
    print(f"{name}: {df.shape}")

print("\nDashboard overview keys:", dashboard_overview.keys())
print("Dashboard asset count:", len(dashboard_assets_manifest["assets"]))
print("Dashboard report count:", len(dashboard_reports_manifest["reports"]))

final_key_results: (8, 4)
final_dataset_summary: (7, 4)
final_model_stack: (4, 8)
final_metric_policy: (6, 4)
final_model_win_summary: (4, 6)
final_uncertainty_summary: (8, 3)
final_sdxl_feasibility: (7, 3)
final_visual_cases: (110, 18)
refined_comparison: (200, 40)
refined_metric_disagreement: (124, 43)
uncertainty_combined_by_case: (40, 55)
uncertainty_vs_refined: (40, 75)

Dashboard overview keys: dict_keys(['generated_at', 'project_title', 'controlled_subset', 'models', 'refined_comparison', 'uncertainty_analysis', 'central_claim', 'main_report'])
Dashboard asset count: 19
Dashboard report count: 3


In [6]:
if dashboard_overview["controlled_subset"]["paintings"] != 50:
    raise ValueError("Expected 50 paintings in dashboard overview.")

if dashboard_overview["controlled_subset"]["damage_cases"] != 250:
    raise ValueError("Expected 250 damage cases in dashboard overview.")

if dashboard_overview["controlled_subset"]["non_zero_comparison_cases"] != 200:
    raise ValueError("Expected 200 non-zero comparison cases in dashboard overview.")

if len(refined_comparison_df) != 200:
    raise ValueError(f"Expected 200 refined comparison rows, found {len(refined_comparison_df)}.")

if len(uncertainty_combined_by_case_df) != 40:
    raise ValueError(
        f"Expected 40 uncertainty cases, found {len(uncertainty_combined_by_case_df)}."
    )

if len(uncertainty_vs_refined_df) != 40:
    raise ValueError(
        f"Expected 40 uncertainty-performance rows, found {len(uncertainty_vs_refined_df)}."
    )

if len(final_key_results_df) != 8:
    raise ValueError(f"Expected 8 final key findings, found {len(final_key_results_df)}.")

if len(final_model_stack_df) != 4:
    raise ValueError(f"Expected 4 model stack rows, found {len(final_model_stack_df)}.")

if len(final_metric_policy_df) != 6:
    raise ValueError(f"Expected 6 metric policy rows, found {len(final_metric_policy_df)}.")

if len(refined_metric_disagreement_df) == 0:
    raise ValueError("Expected non-empty refined metric-disagreement table.")

if final_visual_cases_df["final_figure_exists"].sum() < 10:
    raise ValueError("Expected at least 10 available visual figures.")

expected_fully_evaluated_models = {
    "opencv_telea",
    "lama",
    "stable_diffusion_inpainting",
}

actual_fully_evaluated_models = set(dashboard_overview["models"]["fully_evaluated"])

if actual_fully_evaluated_models != expected_fully_evaluated_models:
    raise ValueError(
        "Fully evaluated model mismatch.\n"
        f"Expected: {expected_fully_evaluated_models}\n"
        f"Found: {actual_fully_evaluated_models}"
    )

if dashboard_overview["models"]["feasibility_audited"] != ["sdxl_inpainting"]:
    raise ValueError("Expected SDXL to be the feasibility-audited model.")

print("Supervisor-package validation gates passed.")
print("Refined disagreement cases:", len(refined_metric_disagreement_df))
print("Visual cases with figures:", int(final_visual_cases_df["final_figure_exists"].sum()))

Supervisor-package validation gates passed.
Refined disagreement cases: 124
Visual cases with figures: 65


In [7]:
def to_project_relative_path(path_value: str | Path) -> str:
    path = Path(str(path_value))

    if not path.is_absolute():
        return path.as_posix()

    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def resolve_project_path(path_value: str | Path) -> Path:
    path = Path(str(path_value))

    if path.is_absolute():
        return path

    return PROJECT_ROOT / path


def write_text_file(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.strip() + "\n", encoding="utf-8")


def write_json_file(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=False)


def copy_file_if_exists(source_path: Path, destination_path: Path) -> bool:
    if not source_path.exists():
        return False

    destination_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_path, destination_path)

    return True


print("Supervisor package helper functions ready.")

Supervisor package helper functions ready.


## Proposal-to-current-work mapping

The supervisor package should explicitly compare the remembered proposal scope against the completed controlled experiment.

The proposal direction was not to train a new restoration model. It was to build a trustworthy evaluation framework using pretrained restoration/inpainting models, synthetic damage, multi-metric evaluation, and uncertainty analysis.

This section creates a research-question coverage table and identifies which points are answered, partially answered, or require supervisor clarification.

In [8]:
proposal_research_questions_df = pd.DataFrame(
    [
        {
            "rq_id": "RQ1",
            "proposal_question": (
                "Can multi-metric evaluation provide a more trustworthy assessment of AI-assisted "
                "painting restoration than relying on PSNR/SSIM or single-score evaluation alone?"
            ),
            "proposal_intent": (
                "Evaluate restoration using complementary metric families such as classical metrics, "
                "perceptual metrics, feature-space similarity, visual diagnostics, and region-aware evaluation."
            ),
            "current_status": "substantially_answered",
        },
        {
            "rq_id": "RQ2",
            "proposal_question": (
                "How do pretrained inpainting/restoration models compare across painting categories "
                "and synthetic damage conditions?"
            ),
            "proposal_intent": (
                "Compare multiple pretrained or established restoration methods across painting styles/categories "
                "and controlled artificial damage types."
            ),
            "current_status": "substantially_answered_for_controlled_50_subset",
        },
        {
            "rq_id": "RQ3",
            "proposal_question": (
                "Can uncertainty estimated from multiple restoration candidates identify cases where "
                "a generative restoration should be treated cautiously?"
            ),
            "proposal_intent": (
                "Use multiple diffusion outputs or candidate restorations to estimate instability, disagreement, "
                "or uncertainty in generated restoration regions."
            ),
            "current_status": "partially_answered_with_balanced_40_case_subset",
        },
    ]
)

proposal_hypotheses_df = pd.DataFrame(
    [
        {
            "hypothesis_id": "H1",
            "hypothesis": (
                "A multi-metric framework will reveal restoration quality differences that are not visible "
                "from PSNR/SSIM alone."
            ),
            "current_evidence": (
                "The final framework combines MSE, PSNR, SSIM, LPIPS, CLIP, DINOv2, visual diagnostics, "
                "metric disagreement cases, and refined metric-region policy."
            ),
            "status": "supported_by_current_experiment",
        },
        {
            "hypothesis_id": "H2",
            "hypothesis": (
                "Pretrained models will behave differently across painting categories and damage types."
            ),
            "current_evidence": (
                "OpenCV Telea, LaMa, and Stable Diffusion were compared over 200 non-zero damage cases, "
                "with summaries by mask type and painting category."
            ),
            "status": "supported_for_current_model_stack",
        },
        {
            "hypothesis_id": "H3",
            "hypothesis": (
                "Variation among multiple diffusion restoration candidates can act as a warning signal "
                "for generative restoration uncertainty."
            ),
            "current_evidence": (
                "Stable Diffusion was evaluated on a balanced 40-case subset using four seeds per case, "
                "producing image-space, LPIPS, CLIP/DINOv2, and combined uncertainty summaries."
            ),
            "status": "supported_diagnostically_but_scope_needs_supervisor_confirmation",
        },
    ]
)

proposal_research_questions_path = (
    supervisor_data_dir / "proposal_research_question_coverage.csv"
)

proposal_hypotheses_path = (
    supervisor_data_dir / "proposal_hypothesis_coverage.csv"
)

proposal_research_questions_df.to_csv(proposal_research_questions_path, index=False)
proposal_hypotheses_df.to_csv(proposal_hypotheses_path, index=False)

print("Saved proposal RQ coverage:", proposal_research_questions_path)
print("Saved proposal hypothesis coverage:", proposal_hypotheses_path)

display(proposal_research_questions_df)
display(proposal_hypotheses_df)

Saved proposal RQ coverage: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\data\proposal_research_question_coverage.csv
Saved proposal hypothesis coverage: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\data\proposal_hypothesis_coverage.csv


,rq_id,proposal_question,proposal_intent,current_status
0,RQ1,Can multi-metric evaluation provide a more tru...,Evaluate restoration using complementary metri...,substantially_answered
1,RQ2,How do pretrained inpainting/restoration model...,Compare multiple pretrained or established res...,substantially_answered_for_controlled_50_subset
2,RQ3,Can uncertainty estimated from multiple restor...,Use multiple diffusion outputs or candidate re...,partially_answered_with_balanced_40_case_subset


,hypothesis_id,hypothesis,current_evidence,status
0,H1,A multi-metric framework will reveal restorati...,"The final framework combines MSE, PSNR, SSIM, ...",supported_by_current_experiment
1,H2,Pretrained models will behave differently acro...,"OpenCV Telea, LaMa, and Stable Diffusion were ...",supported_for_current_model_stack
2,H3,Variation among multiple diffusion restoration...,Stable Diffusion was evaluated on a balanced 4...,supported_diagnostically_but_scope_needs_super...


In [9]:
rq_coverage_rows = [
    {
        "rq_id": "RQ1",
        "proposal_area": "Multi-metric trustworthiness evaluation",
        "implemented_evidence": (
            "Classical metrics, LPIPS, CLIP, DINOv2, error maps, refined metric-region policy, "
            "metric-disagreement export, final consolidated report."
        ),
        "main_outputs": (
            "final_controlled_50_metric_policy_summary.csv; "
            "comparison_unified_refined_opencv_lama_stable_diffusion_50.csv; "
            "comparison_metric_disagreement_cases_refined_opencv_lama_stable_diffusion_50.csv"
        ),
        "answer_status": "strong",
        "remaining_gap": (
            "Need supervisor confirmation that the current metric set is sufficient and that the "
            "metric-region policy is acceptable for thesis framing."
        ),
        "supervisor_question": (
            "Is the refined multi-metric policy acceptable as the final evaluation framework, "
            "especially the decision to evaluate SSIM on mask_bbox_crop rather than sparse masked pixels?"
        ),
    },
    {
        "rq_id": "RQ2",
        "proposal_area": "Pretrained model comparison across painting/damage conditions",
        "implemented_evidence": (
            "OpenCV Telea, LaMa, and Stable Diffusion Inpainting fully evaluated on the controlled "
            "50-painting subset. Summaries generated by mask type and painting category."
        ),
        "main_outputs": (
            "final_controlled_50_model_stack_summary.csv; "
            "final_controlled_50_model_win_summary.csv; "
            "comparison_summary_by_mask_type_refined_opencv_lama_stable_diffusion_50.csv; "
            "comparison_summary_by_category_refined_opencv_lama_stable_diffusion_50.csv"
        ),
        "answer_status": "strong_for_controlled_50_subset",
        "remaining_gap": (
            "Original proposal expected possible scaling toward 300–1000 paintings and 3–5 styles. "
            "The current controlled 50 subset is complete and defensible, but final expected scale "
            "should be clarified."
        ),
        "supervisor_question": (
            "Is the controlled 50-painting subset sufficient for the current thesis checkpoint, "
            "or should the final experiment scale beyond 50 paintings?"
        ),
    },
    {
        "rq_id": "RQ3",
        "proposal_area": "Diffusion uncertainty from multiple candidates",
        "implemented_evidence": (
            "Stable Diffusion Inpainting was sampled with four seeds on a balanced 40-case diagnostic subset. "
            "Image-space uncertainty, LPIPS uncertainty, CLIP/DINOv2 uncertainty, combined uncertainty, "
            "and uncertainty-performance quadrants were generated."
        ),
        "main_outputs": (
            "stable_diffusion_uncertainty_combined_summary_by_case_50.csv; "
            "stable_diffusion_uncertainty_vs_refined_performance_50.csv; "
            "stable_diffusion_uncertainty_performance_quadrants_50.csv"
        ),
        "answer_status": "moderate_to_strong_but_scope_sensitive",
        "remaining_gap": (
            "The uncertainty subset is balanced and methodologically useful, but it covers 40 of 200 "
            "non-zero cases. Supervisor should confirm whether this is sufficient or whether all 200 "
            "non-zero cases should be sampled."
        ),
        "supervisor_question": (
            "Is the 40-case balanced uncertainty subset sufficient, or should Stable Diffusion uncertainty "
            "be expanded to all 200 non-zero damage cases?"
        ),
    },
    {
        "rq_id": "Cross-cutting",
        "proposal_area": "Model feasibility and scope control",
        "implemented_evidence": (
            "SDXL Inpainting was tested but excluded from full local comparison due 6GB VRAM/runtime-quality "
            "constraints. The exclusion is documented as a feasibility limitation rather than a model-quality conclusion."
        ),
        "main_outputs": (
            "final_controlled_50_sdxl_feasibility_summary.csv; "
            "model_audit_notes.md"
        ),
        "answer_status": "requires_supervisor_confirmation",
        "remaining_gap": (
            "Need confirmation whether SDXL feasibility-only treatment is acceptable, or whether university GPU "
            "resources should be requested for a full SDXL comparison."
        ),
        "supervisor_question": (
            "Is it acceptable to keep SDXL as feasibility-audited only, or should university compute be requested "
            "for a full SDXL restoration and four-model comparison?"
        ),
    },
]

rq_coverage_df = pd.DataFrame(rq_coverage_rows)

rq_coverage_path = supervisor_data_dir / "research_question_coverage_summary.csv"
rq_coverage_df.to_csv(rq_coverage_path, index=False)

print("Saved RQ coverage summary:", rq_coverage_path)
display(rq_coverage_df)

Saved RQ coverage summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\data\research_question_coverage_summary.csv


,rq_id,proposal_area,implemented_evidence,main_outputs,answer_status,remaining_gap,supervisor_question
0,RQ1,Multi-metric trustworthiness evaluation,"Classical metrics, LPIPS, CLIP, DINOv2, error ...",final_controlled_50_metric_policy_summary.csv;...,strong,Need supervisor confirmation that the current ...,Is the refined multi-metric policy acceptable ...
1,RQ2,Pretrained model comparison across painting/da...,"OpenCV Telea, LaMa, and Stable Diffusion Inpai...",final_controlled_50_model_stack_summary.csv; f...,strong_for_controlled_50_subset,Original proposal expected possible scaling to...,Is the controlled 50-painting subset sufficien...
2,RQ3,Diffusion uncertainty from multiple candidates,Stable Diffusion Inpainting was sampled with f...,stable_diffusion_uncertainty_combined_summary_...,moderate_to_strong_but_scope_sensitive,The uncertainty subset is balanced and methodo...,Is the 40-case balanced uncertainty subset suf...
3,Cross-cutting,Model feasibility and scope control,SDXL Inpainting was tested but excluded from f...,final_controlled_50_sdxl_feasibility_summary.c...,requires_supervisor_confirmation,Need confirmation whether SDXL feasibility-onl...,Is it acceptable to keep SDXL as feasibility-a...


In [10]:
supervisor_output_paths = {
    "readme": supervisor_package_dir / "README_supervisor.md",
    "methodology_summary": supervisor_package_dir / "methodology_summary.md",
    "results_summary": supervisor_package_dir / "results_summary.md",
    "proposal_alignment": supervisor_package_dir / "proposal_alignment.md",
    "limitations_and_deviations": supervisor_package_dir / "limitations_and_deviations.md",
    "supervisor_questions": supervisor_package_dir / "supervisor_questions.md",
    "next_steps": supervisor_package_dir / "next_steps.md",
    "package_manifest": supervisor_package_dir / "package_manifest.json",
}

for name, path in supervisor_output_paths.items():
    print(f"{name}: {path}")

readme: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\README_supervisor.md
methodology_summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\methodology_summary.md
results_summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\results_summary.md
proposal_alignment: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\proposal_alignment.md
limitations_and_deviations: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\limitations_and_deviations.md
supervisor_questions: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\supervisor_questions.md
next_steps: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\next_steps.md
package_manifest: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\package_manifest.json


## Supervisor package contents

This section creates the actual supervisor review package.

The package includes:

- compact CSV/JSON data summaries,
- selected visual figures,
- main report references/copies,
- proposal alignment notes,
- methodology summary,
- results summary,
- limitations and deviations,
- supervisor questions,
- recommended next steps,
- package manifest.

The package is designed to be readable without opening the full notebook chain.

In [11]:
compact_summary_sources = {
    # Final summaries
    "final_controlled_50_key_results_summary.csv": source_paths["final_key_results"],
    "final_controlled_50_dataset_summary.csv": source_paths["final_dataset_summary"],
    "final_controlled_50_model_stack_summary.csv": source_paths["final_model_stack"],
    "final_controlled_50_metric_policy_summary.csv": source_paths["final_metric_policy"],
    "final_controlled_50_model_win_summary.csv": source_paths["final_model_win_summary"],
    "final_controlled_50_per_metric_winner_summary.csv": source_paths["final_per_metric_winner_summary"],
    "final_controlled_50_uncertainty_summary.csv": source_paths["final_uncertainty_summary"],
    "final_controlled_50_sdxl_feasibility_summary.csv": source_paths["final_sdxl_feasibility"],
    "final_controlled_50_visual_case_summary.csv": source_paths["final_visual_case_summary"],

    # Refined comparison and uncertainty compact summaries
    "comparison_summary_by_mask_type_refined_opencv_lama_stable_diffusion_50.csv": source_paths["refined_summary_by_mask_type"],
    "comparison_summary_by_category_refined_opencv_lama_stable_diffusion_50.csv": source_paths["refined_summary_by_category"],
    "comparison_metric_disagreement_cases_refined_opencv_lama_stable_diffusion_50.csv": source_paths["refined_metric_disagreement"],
    "stable_diffusion_uncertainty_combined_summary_by_mask_type_50.csv": source_paths["uncertainty_by_mask_type"],
    "stable_diffusion_uncertainty_combined_summary_by_category_50.csv": source_paths["uncertainty_by_category"],
    "stable_diffusion_uncertainty_performance_quadrants_50.csv": source_paths["uncertainty_quadrants"],

    # Dashboard manifests
    "dashboard_overview_summary.json": source_paths["dashboard_overview"],
    "dashboard_assets_manifest.json": source_paths["dashboard_assets_manifest"],
    "dashboard_key_findings.json": source_paths["dashboard_key_findings"],
    "dashboard_reports_manifest.json": source_paths["dashboard_reports_manifest"],
}


def robust_copy_file(source_path: Path, destination_path: Path) -> dict:
    source_path = Path(source_path)
    destination_path = Path(destination_path)

    if not source_path.exists():
        return {
            "status": "missing_source",
            "source_path": str(source_path),
            "destination_path": str(destination_path),
            "error": "Source file does not exist.",
        }

    destination_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        if source_path.resolve() == destination_path.resolve():
            return {
                "status": "already_in_place",
                "source_path": str(source_path),
                "destination_path": str(destination_path),
                "error": "",
            }

        if destination_path.exists():
            destination_path.unlink()

        shutil.copy2(source_path, destination_path)

        return {
            "status": "copied",
            "source_path": str(source_path),
            "destination_path": str(destination_path),
            "error": "",
        }

    except PermissionError as error:
        return {
            "status": "permission_error",
            "source_path": str(source_path),
            "destination_path": str(destination_path),
            "error": str(error),
        }

    except Exception as error:
        return {
            "status": "error",
            "source_path": str(source_path),
            "destination_path": str(destination_path),
            "error": repr(error),
        }


copy_results = []

for output_filename, source_path in compact_summary_sources.items():
    destination_path = supervisor_data_dir / output_filename
    copy_results.append(
        robust_copy_file(
            source_path=source_path,
            destination_path=destination_path,
        )
    )

# Generated files from Cells 12 and 13 should already be in supervisor_data_dir.
generated_package_files = {
    "proposal_research_question_coverage.csv": proposal_research_questions_path,
    "proposal_hypothesis_coverage.csv": proposal_hypotheses_path,
    "research_question_coverage_summary.csv": rq_coverage_path,
}

for output_filename, generated_path in generated_package_files.items():
    generated_path = Path(generated_path)

    copy_results.append(
        {
            "status": "already_in_place" if generated_path.exists() else "missing_generated_file",
            "source_path": str(generated_path),
            "destination_path": str(supervisor_data_dir / output_filename),
            "error": "" if generated_path.exists() else "Generated file missing. Rerun Cells 12 and 13.",
        }
    )

copy_results_df = pd.DataFrame(copy_results)

copy_results_path = supervisor_data_dir / "supervisor_compact_summary_copy_results.csv"

try:
    copy_results_df.to_csv(copy_results_path, index=False)
except PermissionError:
    fallback_copy_results_path = (
        supervisor_data_dir
        / f"supervisor_compact_summary_copy_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    )
    copy_results_df.to_csv(fallback_copy_results_path, index=False)
    copy_results_path = fallback_copy_results_path

display(copy_results_df)

failed_copy_results_df = copy_results_df[
    ~copy_results_df["status"].isin(["copied", "already_in_place"])
].copy()

if not failed_copy_results_df.empty:
    print("Some compact summary files could not be prepared:")
    display(failed_copy_results_df)

    if (failed_copy_results_df["status"] == "missing_generated_file").any():
        raise FileNotFoundError(
            "Generated proposal/RQ coverage files are missing. "
            "Rerun Cells 12 and 13, then rerun this cell."
        )

    if (failed_copy_results_df["status"] == "permission_error").any():
        raise PermissionError(
            "At least one supervisor-package summary file is locked. "
            "Close File Explorer preview panes, Excel, VS Code tabs, or PowerShell sessions inside "
            "outputs/supervisor_package/data, then rerun this cell."
        )

    raise RuntimeError("Some supervisor-package summary files failed to copy.")

copied_summary_files = [
    Path(row["destination_path"])
    for _, row in copy_results_df.iterrows()
    if row["status"] in ["copied", "already_in_place"]
]

print("Prepared compact supervisor summary files:", len(copied_summary_files))
print("Copy/preparation log:", copy_results_path)

for path in copied_summary_files:
    print(to_project_relative_path(path))

,status,source_path,destination_path,error
0,copied,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,
1,copied,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,
2,copied,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,
3,copied,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,
4,copied,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,
5,copied,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,
6,copied,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,
7,copied,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,
8,copied,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,
9,copied,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,


Prepared compact supervisor summary files: 22
Copy/preparation log: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\data\supervisor_compact_summary_copy_results.csv
outputs/supervisor_package/data/final_controlled_50_key_results_summary.csv
outputs/supervisor_package/data/final_controlled_50_dataset_summary.csv
outputs/supervisor_package/data/final_controlled_50_model_stack_summary.csv
outputs/supervisor_package/data/final_controlled_50_metric_policy_summary.csv
outputs/supervisor_package/data/final_controlled_50_model_win_summary.csv
outputs/supervisor_package/data/final_controlled_50_per_metric_winner_summary.csv
outputs/supervisor_package/data/final_controlled_50_uncertainty_summary.csv
outputs/supervisor_package/data/final_controlled_50_sdxl_feasibility_summary.csv
outputs/supervisor_package/data/final_controlled_50_visual_case_summary.csv
outputs/supervisor_package/data/comparison_summary_by_mask_type_refined_opencv_lama_stable_diffusion_50.csv
outputs/su

In [12]:
available_visual_cases_df = final_visual_cases_df[
    final_visual_cases_df["final_figure_exists"] == True
].copy()

if available_visual_cases_df.empty:
    raise ValueError("No available visual cases found for supervisor package.")

# Select a small, readable set rather than dumping 100+ figures.
selected_visual_rows = []

selection_plan = [
    ("refined_model_comparison", 6),
    ("stable_diffusion_uncertainty", 6),
    ("uncertainty_trustworthiness_quadrant", 8),
]

for source_label, max_rows in selection_plan:
    subset_df = available_visual_cases_df[
        available_visual_cases_df["final_visual_source"] == source_label
    ].copy()

    if subset_df.empty:
        continue

    if "combined_uncertainty_index" in subset_df.columns:
        subset_df = subset_df.sort_values(
            "combined_uncertainty_index",
            ascending=False,
            na_position="last",
        )

    selected_visual_rows.append(subset_df.head(max_rows))

selected_supervisor_visual_cases_df = pd.concat(
    selected_visual_rows,
    ignore_index=True,
) if selected_visual_rows else available_visual_cases_df.head(12).copy()

# Deduplicate figure paths, keeping the first descriptive row.
selected_supervisor_visual_cases_df = (
    selected_supervisor_visual_cases_df
    .drop_duplicates(subset=["final_figure_path"])
    .reset_index(drop=True)
)

selected_supervisor_visual_cases_path = (
    supervisor_data_dir / "selected_supervisor_visual_cases.csv"
)

copied_figure_records = []

for index, row in selected_supervisor_visual_cases_df.iterrows():
    source_figure_path = resolve_project_path(row["final_figure_path"])

    if not source_figure_path.exists():
        continue

    safe_case_id = str(row.get("original_case_id", f"case_{index}")).replace("/", "_").replace("\\", "_")
    safe_source = str(row.get("final_visual_source", "visual")).replace("/", "_").replace("\\", "_")

    destination_filename = f"{index + 1:02d}_{safe_source}_{safe_case_id}{source_figure_path.suffix}"
    destination_path = supervisor_figures_dir / destination_filename

    shutil.copy2(source_figure_path, destination_path)

    copied_figure_records.append(
        {
            "original_case_id": row.get("original_case_id", ""),
            "category": row.get("category", ""),
            "mask_type": row.get("mask_type", ""),
            "visual_source": row.get("final_visual_source", ""),
            "visual_reason": row.get("final_visual_reason", ""),
            "source_figure_path": to_project_relative_path(source_figure_path),
            "package_figure_path": to_project_relative_path(destination_path),
        }
    )

selected_supervisor_visual_manifest_df = pd.DataFrame(copied_figure_records)

selected_supervisor_visual_manifest_path = (
    supervisor_data_dir / "selected_supervisor_visual_manifest.csv"
)

selected_supervisor_visual_manifest_df.to_csv(
    selected_supervisor_visual_manifest_path,
    index=False,
)

selected_supervisor_visual_cases_df.to_csv(
    selected_supervisor_visual_cases_path,
    index=False,
)

print("Selected supervisor visual cases:", len(selected_supervisor_visual_cases_df))
print("Copied supervisor figures:", len(selected_supervisor_visual_manifest_df))
print("Saved visual manifest:", selected_supervisor_visual_manifest_path)

display(selected_supervisor_visual_manifest_df.head(20))

Selected supervisor visual cases: 18
Copied supervisor figures: 18
Saved visual manifest: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\data\selected_supervisor_visual_manifest.csv


,original_case_id,category,mask_type,visual_source,visual_reason,source_figure_path,package_figure_path
0,p004_scratch_thin,portrait_figure,scratch_thin,refined_model_comparison,old_vs_refined_vote_changed,outputs/figures/model_comparison/opencv_lama_s...,outputs/supervisor_package/selected_figures/01...
1,p040_loss_large,abstraction_surrealism,loss_large,refined_model_comparison,old_vs_refined_vote_changed; refined_metric_di...,outputs/figures/model_comparison/opencv_lama_s...,outputs/supervisor_package/selected_figures/02...
2,p001_mixed_damage,portrait_figure,mixed_damage,refined_model_comparison,old_vs_refined_vote_changed; refined_metric_di...,outputs/figures/model_comparison/opencv_lama_s...,outputs/supervisor_package/selected_figures/03...
3,p030_loss_large,architecture_structured,loss_large,refined_model_comparison,refined_category_mask_representative,outputs/figures/model_comparison/opencv_lama_s...,outputs/supervisor_package/selected_figures/04...
4,p001_loss_large,portrait_figure,loss_large,refined_model_comparison,refined_category_mask_representative; refined_...,outputs/figures/model_comparison/opencv_lama_s...,outputs/supervisor_package/selected_figures/05...
5,p037_scratch_thin,abstraction_surrealism,scratch_thin,refined_model_comparison,refined_category_mask_representative; refined_...,outputs/figures/model_comparison/opencv_lama_s...,outputs/supervisor_package/selected_figures/06...
6,p031_loss_large,abstraction_surrealism,loss_large,stable_diffusion_uncertainty,category_mask_representative_highest,outputs/figures/uncertainty/stable_diffusion_i...,outputs/supervisor_package/selected_figures/07...
7,p037_loss_small,abstraction_surrealism,loss_small,stable_diffusion_uncertainty,category_mask_representative_highest,outputs/figures/uncertainty/stable_diffusion_i...,outputs/supervisor_package/selected_figures/08...
8,p037_mixed_damage,abstraction_surrealism,mixed_damage,stable_diffusion_uncertainty,category_mask_representative_highest,outputs/figures/uncertainty/stable_diffusion_i...,outputs/supervisor_package/selected_figures/09...
9,p037_scratch_thin,abstraction_surrealism,scratch_thin,stable_diffusion_uncertainty,category_mask_representative_highest,outputs/figures/uncertainty/stable_diffusion_i...,outputs/supervisor_package/selected_figures/10...


In [13]:
# Copy main reports. These may be large, but the supervisor package should contain or at least reference them.
report_copy_sources = {
    "final_controlled_50_evaluation_report.html": source_paths["final_report"],
    "opencv_lama_stable_diffusion_refined_metric_comparison_report_50.html": source_paths["refined_comparison_report"],
    "stable_diffusion_uncertainty_report_50.html": source_paths["uncertainty_report"],
}

copied_report_files = []

for output_filename, source_path in report_copy_sources.items():
    destination_path = supervisor_reports_dir / output_filename

    if copy_file_if_exists(source_path, destination_path):
        copied_report_files.append(destination_path)
    else:
        print(f"Could not copy report: {source_path}")

doc_copy_sources = {
    "methodology_notes.md": source_paths["methodology_notes"],
    "literature_reference_log.md": source_paths["literature_reference_log"],
    "model_audit_notes.md": source_paths["model_audit_notes"],
}

copied_doc_files = []

for output_filename, source_path in doc_copy_sources.items():
    destination_path = supervisor_docs_dir / output_filename

    if copy_file_if_exists(source_path, destination_path):
        copied_doc_files.append(destination_path)
    else:
        print(f"Could not copy doc: {source_path}")

print("Copied reports:", len(copied_report_files))
for path in copied_report_files:
    print(to_project_relative_path(path), f"({path.stat().st_size / (1024 * 1024):.2f} MB)")

print("\nCopied docs:", len(copied_doc_files))
for path in copied_doc_files:
    print(to_project_relative_path(path))

Copied reports: 3
outputs/supervisor_package/reports/final_controlled_50_evaluation_report.html (192.95 MB)
outputs/supervisor_package/reports/opencv_lama_stable_diffusion_refined_metric_comparison_report_50.html (24.68 MB)
outputs/supervisor_package/reports/stable_diffusion_uncertainty_report_50.html (185.91 MB)

Copied docs: 3
outputs/supervisor_package/docs/methodology_notes.md
outputs/supervisor_package/docs/literature_reference_log.md
outputs/supervisor_package/docs/model_audit_notes.md


In [14]:
report_size_rows = []

for report_path in copied_report_files:
    report_size_rows.append(
        {
            "report": report_path.name,
            "package_path": to_project_relative_path(report_path),
            "size_mb": report_path.stat().st_size / (1024 * 1024),
            "github_warning": report_path.stat().st_size > 50 * 1024 * 1024,
            "github_hard_limit_risk": report_path.stat().st_size > 100 * 1024 * 1024,
        }
    )

supervisor_report_size_df = pd.DataFrame(report_size_rows)

supervisor_report_size_path = supervisor_data_dir / "supervisor_report_file_sizes.csv"
supervisor_report_size_df.to_csv(supervisor_report_size_path, index=False)

display(supervisor_report_size_df)

if supervisor_report_size_df["github_hard_limit_risk"].any():
    print(
        "WARNING: At least one copied report exceeds 100 MB. "
        "Do not expect GitHub normal push to accept it. "
        "Use linked reports, external storage, or Git LFS if needed."
    )
else:
    print("No copied report exceeds 100 MB.")

,report,package_path,size_mb,github_warning,github_hard_limit_risk
0,final_controlled_50_evaluation_report.html,outputs/supervisor_package/reports/final_contr...,192.954260,True,True
1,opencv_lama_stable_diffusion_refined_metric_co...,outputs/supervisor_package/reports/opencv_lama...,24.678207,False,False
2,stable_diffusion_uncertainty_report_50.html,outputs/supervisor_package/reports/stable_diff...,185.914966,True,True


In [15]:
package_generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

overview = dashboard_overview
refined_overview = overview["refined_comparison"]
uncertainty_overview = overview["uncertainty_analysis"]

readme_content = f"""
# Supervisor Review Package

Generated: {package_generated_at}

Project:

**Trustworthy Evaluation Frameworks for AI-Assisted Painting Restoration**

## Purpose of this package

This package summarizes the current state of the thesis experiment without requiring review of every notebook and intermediate artifact.

The work is framed as an **evaluation framework**, not as a proposal for a new restoration model.

The central thesis claim is:

> Visual plausibility is not the same as restoration trustworthiness.

## Current experiment status

The controlled evaluation currently includes:

- {overview["controlled_subset"]["paintings"]} paintings,
- {overview["controlled_subset"]["painting_categories"]} painting categories,
- {overview["controlled_subset"]["damage_cases"]} total synthetic damage cases,
- {overview["controlled_subset"]["non_zero_comparison_cases"]} non-zero restoration comparison cases.

Fully evaluated models:

- OpenCV Telea,
- LaMa,
- Stable Diffusion Inpainting.

Feasibility-audited model:

- SDXL Inpainting.

## Main quantitative finding

Under the refined metric-region comparison:

- LaMa majority-vote wins: {refined_overview["lama_majority_cases"]}/{refined_overview["total_non_zero_cases"]},
- OpenCV Telea majority-vote wins: {refined_overview["opencv_telea_majority_cases"]}/{refined_overview["total_non_zero_cases"]},
- Stable Diffusion Inpainting majority-vote wins: {refined_overview["stable_diffusion_inpainting_majority_cases"]}/{refined_overview["total_non_zero_cases"]}.

This supports the current interpretation that LaMa is strongest under reference-based metrics, while Stable Diffusion may produce visually plausible but less reference-faithful restorations.

## Stable Diffusion uncertainty

Stable Diffusion was evaluated with multi-seed uncertainty analysis:

- {uncertainty_overview["cases"]} balanced diagnostic cases,
- {uncertainty_overview["seed_outputs"]} generated seed outputs,
- {uncertainty_overview["seeds_per_case"]} seeds per case.

The highest uncertainty case in the current subset is:

- `{uncertainty_overview["highest_uncertainty_case"]}`,
- mask type: `{uncertainty_overview["highest_uncertainty_case_mask_type"]}`,
- combined uncertainty index: {uncertainty_overview["highest_uncertainty_index"]:.4f}.

## SDXL status

SDXL Inpainting was tested locally but excluded from full local evaluation because the available 6GB GPU did not provide a practical runtime-quality balance.

This is documented as a computational feasibility limitation, not as a full model-quality conclusion about SDXL.

## Important files in this package

### Main summaries

- `data/final_controlled_50_key_results_summary.csv`
- `data/research_question_coverage_summary.csv`
- `data/final_controlled_50_model_stack_summary.csv`
- `data/final_controlled_50_metric_policy_summary.csv`
- `data/final_controlled_50_model_win_summary.csv`
- `data/final_controlled_50_uncertainty_summary.csv`
- `data/final_controlled_50_sdxl_feasibility_summary.csv`

### Main reports

- `reports/final_controlled_50_evaluation_report.html`
- `reports/opencv_lama_stable_diffusion_refined_metric_comparison_report_50.html`
- `reports/stable_diffusion_uncertainty_report_50.html`

### Supervisor-facing notes

- `proposal_alignment.md`
- `methodology_summary.md`
- `results_summary.md`
- `limitations_and_deviations.md`
- `supervisor_questions.md`
- `next_steps.md`

### Selected figures

- `selected_figures/`

## Requested supervisor feedback

The most important feedback points are:

1. Whether the current research-question coverage is sufficient.
2. Whether the 50-painting controlled subset is sufficient for the next thesis checkpoint.
3. Whether the 40-case Stable Diffusion uncertainty subset is sufficient or should be expanded.
4. Whether SDXL feasibility-only treatment is acceptable.
5. Whether the refined metric-region policy is acceptable.
6. Whether the thesis framing should emphasize evaluation-framework trustworthiness rather than model ranking.
"""

write_text_file(supervisor_output_paths["readme"], readme_content)

print("Saved README:", supervisor_output_paths["readme"])

Saved README: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\README_supervisor.md


In [16]:
rq_table_md = rq_coverage_df.to_markdown(index=False)
hypothesis_table_md = proposal_hypotheses_df.to_markdown(index=False)

proposal_alignment_content = f"""
# Proposal Alignment

This note maps the completed controlled experiment back to the remembered thesis proposal framing.

## Original proposal direction

The thesis was framed around a trustworthy evaluation framework for AI-assisted painting restoration.

The work was not intended to train a new restoration model. Instead, the planned contribution was an evaluation framework using:

- controlled painting data,
- synthetic damage,
- pretrained restoration/inpainting models,
- multiple metric families,
- visual diagnostics,
- uncertainty analysis.

## Research question coverage

{rq_table_md}

## Hypothesis coverage

{hypothesis_table_md}

## Current interpretation

The current experiment substantially answers RQ1 and RQ2 for the controlled 50-painting subset.

RQ3 is answered diagnostically through the 40-case Stable Diffusion uncertainty subset, but its final scope should be confirmed with the supervisor.

## Main proposal alignment conclusion

The current work remains aligned with the proposal because it demonstrates a reproducible evaluation framework rather than a new restoration model.

The strongest current contribution is the layered evaluation design:

1. controlled synthetic damage,
2. model comparison across multiple restoration paradigms,
3. refined region-aware metrics,
4. metric-disagreement analysis,
5. Stable Diffusion uncertainty diagnostics,
6. SDXL feasibility auditing.

## Main clarification needed

The main remaining clarification is scale:

- Is the current controlled 50-painting subset sufficient for the thesis checkpoint?
- Should the final thesis scale toward a larger dataset?
- Should uncertainty be expanded from 40 diagnostic cases to all 200 non-zero cases?
- Should SDXL be revisited on university compute?
"""

write_text_file(supervisor_output_paths["proposal_alignment"], proposal_alignment_content)

print("Saved proposal alignment:", supervisor_output_paths["proposal_alignment"])

Saved proposal alignment: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\proposal_alignment.md


In [17]:
metric_policy_table_md = final_metric_policy_df.to_markdown(index=False)
model_stack_table_md = final_model_stack_df.to_markdown(index=False)

methodology_summary_content = f"""
# Methodology Summary

## Dataset

The current controlled benchmark contains:

- 50 paintings,
- 5 painting categories,
- 10 paintings per category,
- 5 mask types per painting,
- 250 total damage cases,
- 200 non-zero local comparison cases.

The five painting categories are:

- portrait_figure,
- landscape_natural,
- architecture_structured,
- abstraction_surrealism,
- high_texture_brushwork.

## Synthetic damage

Five mask conditions are used:

- zero_control,
- scratch_thin,
- loss_small,
- loss_large,
- mixed_damage.

The zero-control cases are used for sanity checking. The four non-zero mask types are used for restoration comparison.

## Evaluated model stack

{model_stack_table_md}

## Final metric-region policy

{metric_policy_table_md}

## Evaluation framework

The framework combines:

- classical full-reference metrics,
- perceptual LPIPS metrics,
- CLIP feature-space similarity,
- DINOv2 feature-space similarity,
- visual error-map diagnostics,
- metric-disagreement analysis,
- Stable Diffusion multi-seed uncertainty,
- SDXL feasibility auditing.

## Important methodological decision

Sparse masked-region SSIM was found to be invalid for local comparison because SSIM requires local image structure.

Final policy:

- MSE and PSNR remain on the sparse masked region.
- SSIM is evaluated on the mask-bounding-box crop.
- LPIPS, CLIP, and DINOv2 are also evaluated on the mask-bounding-box crop.

## Interpretation boundary

The project does not claim that model outputs are historically correct painting restorations.

The outputs are evaluated as candidate restorations under controlled synthetic damage with known clean references.

The purpose is to evaluate restoration trustworthiness, not to certify conservation-ready restoration.
"""

write_text_file(supervisor_output_paths["methodology_summary"], methodology_summary_content)

print("Saved methodology summary:", supervisor_output_paths["methodology_summary"])

Saved methodology summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\methodology_summary.md


In [18]:
key_results_table_md = final_key_results_df.to_markdown(index=False)
model_win_table_md = final_model_win_summary_df.to_markdown(index=False)
uncertainty_quadrants_table_md = uncertainty_quadrants_df.to_markdown(index=False)

results_summary_content = f"""
# Results Summary

## Main thesis claim

> Visual plausibility is not the same as restoration trustworthiness.

## Key findings

{key_results_table_md}

## Refined model comparison

The final refined comparison includes 200 non-zero damage cases.

Model win summary:

{model_win_table_md}

Main interpretation:

- LaMa dominates the refined reference-based metric comparison.
- OpenCV Telea remains useful as a deterministic baseline.
- Stable Diffusion Inpainting rarely wins under reference-based majority vote.
- Stable Diffusion remains important because it exposes generative plausibility and uncertainty issues.

## Metric disagreement

The refined comparison exported {len(refined_metric_disagreement_df)} metric-disagreement cases.

This is important because the thesis is not simply a leaderboard. Metric disagreement supports the framework argument that restoration trustworthiness depends on multiple complementary signals.

## Stable Diffusion uncertainty

Stable Diffusion uncertainty was evaluated on a balanced 40-case subset with four seeds per case.

Outputs:

- 40 uncertainty cases,
- 160 generated outputs,
- 240 pairwise LPIPS comparisons,
- 240 pairwise CLIP/DINOv2 feature comparisons.

Uncertainty-performance quadrants:

{uncertainty_quadrants_table_md}

Main interpretation:

Stable Diffusion uncertainty provides a diagnostic warning signal. A restoration can appear visually plausible while being unstable across seeds or weak under reference-based metrics.

## SDXL feasibility

SDXL was excluded from full local comparison because local 6GB VRAM and runtime constraints made full evaluation impractical.

This is a feasibility limitation, not a model-quality conclusion.

## Overall result

The current experiment supports the thesis framing that trustworthy AI-assisted painting restoration evaluation requires:

- reference-based metrics,
- region-aware metric policy,
- model comparison,
- visual diagnostics,
- uncertainty analysis for generative models,
- explicit model feasibility and audit notes.
"""

write_text_file(supervisor_output_paths["results_summary"], results_summary_content)

print("Saved results summary:", supervisor_output_paths["results_summary"])

Saved results summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\results_summary.md


In [19]:
sdxl_table_md = final_sdxl_feasibility_df.to_markdown(index=False)

limitations_content = f"""
# Limitations and Deviations from Proposal

## 1. Dataset scale

The completed controlled evaluation uses 50 paintings.

This is smaller than the broader long-term target range discussed during thesis planning, where 300–1000 paintings were considered as a possible later scale.

Current interpretation:

- The 50-painting subset is balanced and complete.
- It covers five painting categories and five damage conditions.
- It is sufficient for a controlled methodological demonstration.
- Supervisor confirmation is needed on whether this is enough for the final thesis scope or only for the current checkpoint.

## 2. SDXL feasibility limitation

SDXL was initially considered as a fourth model candidate.

Current SDXL status:

{sdxl_table_md}

Interpretation:

SDXL was excluded from the full local comparison because the available local GPU could not support practical full evaluation.

This is not a claim that SDXL performs poorly under adequate compute. It is a documented feasibility limitation.

## 3. Stable Diffusion uncertainty subset

Stable Diffusion uncertainty was evaluated on a balanced 40-case diagnostic subset.

This subset includes:

- 5 painting categories,
- 2 paintings per category,
- 4 non-zero masks per selected painting,
- 4 seeds per case.

This produced 160 generated outputs.

Current interpretation:

- The subset is balanced and useful for diagnostic uncertainty analysis.
- It does not cover all 200 non-zero comparison cases.
- Supervisor confirmation is needed on whether the subset is sufficient or should be expanded.

## 4. Synthetic damage limitation

The experiment uses controlled synthetic damage rather than real restoration ground truth.

This is appropriate because the clean reference is known, enabling full-reference metrics.

However, synthetic damage does not fully represent real physical deterioration, conservation constraints, pigment aging, varnish changes, craquelure, or historical restoration complexity.

## 5. Metric limitations

The metric framework includes multiple complementary metrics, but none of them individually determines conservation validity.

Metric limitations include:

- MSE/PSNR reward pixel-level closeness but may not capture perceptual quality.
- SSIM needs image-like spatial regions and is not valid on sparse masked pixels.
- LPIPS is perceptual but not painting-conservation-specific.
- CLIP and DINOv2 are general pretrained feature spaces, not restoration-faithfulness judges.
- Visual plausibility may not equal reference faithfulness.

## 6. Model-domain limitation

OpenCV Telea, LaMa, Stable Diffusion Inpainting, and SDXL Inpainting are not painting-conservation-specific restoration systems.

LaMa and Stable Diffusion rely on general inpainting or generative priors.

This creates a domain gap for paintings, historical style, brushwork, abstraction, and conservation interpretation.

## 7. Dashboard status

Dashboard assets have been prepared under `outputs/dashboard/`.

The actual Streamlit dashboard interface is not yet built or updated.

Supervisor confirmation is useful before investing time in polishing the dashboard, especially regarding whether it should be submitted as a formal supporting artifact.
"""

write_text_file(supervisor_output_paths["limitations_and_deviations"], limitations_content)

print("Saved limitations and deviations:", supervisor_output_paths["limitations_and_deviations"])

Saved limitations and deviations: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\limitations_and_deviations.md


In [20]:
supervisor_questions = [
    {
        "priority": "high",
        "question": (
            "Is the current 50-painting controlled subset sufficient for the current thesis checkpoint, "
            "or should the final thesis experiment scale beyond 50 paintings?"
        ),
        "why_it_matters": (
            "The current subset is balanced and complete, but earlier planning mentioned possible scaling "
            "toward 300–1000 paintings."
        ),
        "related_rq": "RQ2",
    },
    {
        "priority": "high",
        "question": (
            "Is the 40-case Stable Diffusion uncertainty subset sufficient, or should uncertainty be expanded "
            "to all 200 non-zero damage cases?"
        ),
        "why_it_matters": (
            "The current uncertainty analysis is balanced and diagnostic, but it does not cover every non-zero case."
        ),
        "related_rq": "RQ3",
    },
    {
        "priority": "high",
        "question": (
            "Is it acceptable to keep SDXL as feasibility-audited only, or should university GPU resources "
            "be requested for a full SDXL restoration and four-model comparison?"
        ),
        "why_it_matters": (
            "SDXL was planned as a possible higher-capacity diffusion comparison, but local 6GB VRAM made full "
            "evaluation impractical."
        ),
        "related_rq": "RQ2/RQ3",
    },
    {
        "priority": "high",
        "question": (
            "Is the refined metric-region policy acceptable, especially the decision to move SSIM from sparse "
            "masked_region to mask_bbox_crop?"
        ),
        "why_it_matters": (
            "The initial sparse masked-region SSIM comparison was invalid. The refined policy keeps SSIM but "
            "evaluates it on an image-like crop."
        ),
        "related_rq": "RQ1",
    },
    {
        "priority": "medium",
        "question": (
            "Is the thesis framing clear enough as an evaluation framework rather than as a model-training or "
            "restoration-production thesis?"
        ),
        "why_it_matters": (
            "The strongest current contribution is the trustworthiness evaluation design, not a new restoration model."
        ),
        "related_rq": "all",
    },
    {
        "priority": "medium",
        "question": (
            "Are the five synthetic damage types sufficient for the thesis scope?"
        ),
        "why_it_matters": (
            "The current masks cover zero control, scratches, small losses, large losses, and mixed damage, "
            "but real painting damage is more complex."
        ),
        "related_rq": "RQ2",
    },
    {
        "priority": "medium",
        "question": (
            "Should the final thesis emphasize the LaMa versus Stable Diffusion contrast as the main empirical result?"
        ),
        "why_it_matters": (
            "LaMa dominates reference metrics, while Stable Diffusion illustrates visual plausibility and uncertainty issues."
        ),
        "related_rq": "RQ1/RQ2/RQ3",
    },
    {
        "priority": "medium",
        "question": (
            "Should the Streamlit dashboard be included as a formal supporting artifact or kept as an internal demo?"
        ),
        "why_it_matters": (
            "Dashboard assets are prepared, but the final app should reflect the supervisor-approved story."
        ),
        "related_rq": "all",
    },
    {
        "priority": "low",
        "question": (
            "Should the final report include full embedded HTML reports, or should large reports be kept as local artifacts "
            "with smaller linked versions for GitHub?"
        ),
        "why_it_matters": (
            "Some reports and notebooks may be too large for clean GitHub storage."
        ),
        "related_rq": "reproducibility",
    },
]

supervisor_questions_df = pd.DataFrame(supervisor_questions)
supervisor_questions_csv_path = supervisor_data_dir / "supervisor_questions.csv"
supervisor_questions_df.to_csv(supervisor_questions_csv_path, index=False)

question_sections = []

for priority, group_df in supervisor_questions_df.groupby("priority", sort=False):
    section_lines = [f"## {priority.capitalize()} priority questions"]

    for index, row in group_df.iterrows():
        section_lines.append(
            f"""
### {index + 1}. {row['question']}

**Related research question:** {row['related_rq']}

**Why this matters:** {row['why_it_matters']}
"""
        )

    question_sections.append("\n".join(section_lines))

supervisor_questions_content = f"""
# Supervisor Questions

The questions below are intended to guide the next supervisor discussion.

They focus on whether the current work sufficiently answers the proposal research questions and whether any additional final experiments are needed.

{chr(10).join(question_sections)}
"""

write_text_file(supervisor_output_paths["supervisor_questions"], supervisor_questions_content)

print("Saved supervisor questions markdown:", supervisor_output_paths["supervisor_questions"])
print("Saved supervisor questions CSV:", supervisor_questions_csv_path)

display(supervisor_questions_df)

Saved supervisor questions markdown: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\supervisor_questions.md
Saved supervisor questions CSV: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\data\supervisor_questions.csv


,priority,question,why_it_matters,related_rq
0,high,Is the current 50-painting controlled subset s...,"The current subset is balanced and complete, b...",RQ2
1,high,Is the 40-case Stable Diffusion uncertainty su...,The current uncertainty analysis is balanced a...,RQ3
2,high,Is it acceptable to keep SDXL as feasibility-a...,SDXL was planned as a possible higher-capacity...,RQ2/RQ3
3,high,Is the refined metric-region policy acceptable...,The initial sparse masked-region SSIM comparis...,RQ1
4,medium,Is the thesis framing clear enough as an evalu...,The strongest current contribution is the trus...,all
5,medium,Are the five synthetic damage types sufficient...,"The current masks cover zero control, scratche...",RQ2
6,medium,Should the final thesis emphasize the LaMa ver...,"LaMa dominates reference metrics, while Stable...",RQ1/RQ2/RQ3
7,medium,Should the Streamlit dashboard be included as ...,"Dashboard assets are prepared, but the final a...",all
8,low,Should the final report include full embedded ...,Some reports and notebooks may be too large fo...,reproducibility


In [21]:
next_steps_content = """
# Recommended Next Steps

## Immediate next step

Review this supervisor package and discuss the open questions with the supervisor.

The most important decisions are:

1. whether the current 50-painting controlled subset is sufficient,
2. whether the 40-case Stable Diffusion uncertainty subset is sufficient,
3. whether SDXL should remain feasibility-audited or be revisited on university compute,
4. whether the refined metric-region policy is accepted,
5. whether the dashboard should be included as a formal artifact.

## If the supervisor accepts the current experimental scope

Proceed with:

1. building or updating the Streamlit dashboard using `outputs/dashboard/`,
2. preparing thesis-ready methods/results assets,
3. drafting the methodology chapter,
4. drafting the results chapter,
5. drafting the limitations and future work section.

## If the supervisor asks for larger uncertainty coverage

Create an additional notebook:

`27b_full_stable_diffusion_uncertainty_sweep_cleaned.ipynb`

Recommended scope:

- all 200 non-zero Stable Diffusion cases,
- 4 seeds per case,
- approximately 800 generated outputs.

This would extend the current 40-case diagnostic uncertainty subset.

## If the supervisor asks for SDXL comparison

Request university or external compute.

Minimum recommended hardware:

- 12GB VRAM minimum,
- 16GB+ VRAM preferred.

Then create optional remote-compute notebooks:

- `32_sdxl_full_restoration_remote_cleaned.ipynb`,
- `33_sdxl_metrics_remote_cleaned.ipynb`,
- `34_four_model_comparison_remote_cleaned.ipynb`.

## If the supervisor asks for dataset scaling

Extend the controlled dataset beyond 50 paintings.

Suggested path:

1. preserve the current 50-painting subset as a validated benchmark,
2. scale preprocessing and masks to a larger dataset,
3. rerun feasible models first,
4. treat heavier diffusion and uncertainty work selectively if compute is limited.

## Dashboard next step

The dashboard should be built after supervisor feedback.

Reason:

The dashboard should reflect the approved thesis story. Building the UI before confirming the framing risks polishing the wrong narrative.

The dashboard should use only:

`outputs/dashboard/`

and should avoid loading raw experimental files directly.

## Thesis asset next step

After the dashboard decision, create:

`31_thesis_methods_assets_cleaned.ipynb`

This notebook should generate thesis-ready:

- tables,
- figures,
- captions,
- methodology snippets,
- results snippets,
- limitations snippets,
- reproducibility notes.
"""

write_text_file(supervisor_output_paths["next_steps"], next_steps_content)

print("Saved next steps:", supervisor_output_paths["next_steps"])

Saved next steps: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\next_steps.md


## Final supervisor package validation

This section creates the package manifest and validates that the supervisor package contains the expected files.

The package should include:

- supervisor-facing markdown summaries,
- compact CSV/JSON data summaries,
- selected visual figures,
- copied project documentation,
- report references or report copies,
- final package manifest.

In [22]:
def collect_package_files(root_dir: Path) -> list[dict]:
    records = []

    for path in sorted(root_dir.rglob("*")):
        if path.is_file():
            records.append(
                {
                    "relative_path": path.relative_to(root_dir).as_posix(),
                    "project_relative_path": to_project_relative_path(path),
                    "file_type": path.suffix.lower().replace(".", ""),
                    "size_bytes": path.stat().st_size,
                    "size_mb": path.stat().st_size / (1024 * 1024),
                }
            )

    return records


package_file_records = collect_package_files(supervisor_package_dir)

package_manifest = {
    "generated_at": package_generated_at,
    "package_root": to_project_relative_path(supervisor_package_dir),
    "project_title": "Trustworthy Evaluation Frameworks for AI-Assisted Painting Restoration",
    "central_claim": "Visual plausibility is not the same as restoration trustworthiness.",
    "controlled_subset": dashboard_overview["controlled_subset"],
    "models": dashboard_overview["models"],
    "refined_comparison": dashboard_overview["refined_comparison"],
    "uncertainty_analysis": dashboard_overview["uncertainty_analysis"],
    "package_sections": {
        "data": to_project_relative_path(supervisor_data_dir),
        "selected_figures": to_project_relative_path(supervisor_figures_dir),
        "reports": to_project_relative_path(supervisor_reports_dir),
        "docs": to_project_relative_path(supervisor_docs_dir),
    },
    "files": package_file_records,
}

write_json_file(supervisor_output_paths["package_manifest"], package_manifest)

package_manifest_df = pd.DataFrame(package_file_records)
package_manifest_csv_path = supervisor_data_dir / "package_manifest_files.csv"
package_manifest_df.to_csv(package_manifest_csv_path, index=False)

print("Saved package manifest JSON:", supervisor_output_paths["package_manifest"])
print("Saved package manifest CSV:", package_manifest_csv_path)
print("Total package files:", len(package_file_records))
print("Total package size MB:", package_manifest_df["size_mb"].sum().round(2))

display(package_manifest_df.sort_values("size_mb", ascending=False).head(20))

Saved package manifest JSON: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\package_manifest.json
Saved package manifest CSV: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\data\package_manifest_files.csv
Total package files: 58
Total package size MB: 473.01


,relative_path,project_relative_path,file_type,size_bytes,size_mb
35,reports/final_controlled_50_evaluation_report....,outputs/supervisor_package/reports/final_contr...,html,202327206,192.954260
37,reports/stable_diffusion_uncertainty_report_50...,outputs/supervisor_package/reports/stable_diff...,html,194945971,185.914966
36,reports/opencv_lama_stable_diffusion_refined_m...,outputs/supervisor_package/reports/opencv_lama...,html,25876976,24.678207
45,selected_figures/07_stable_diffusion_uncertain...,outputs/supervisor_package/selected_figures/07...,png,6341385,6.047616
55,selected_figures/17_uncertainty_trustworthines...,outputs/supervisor_package/selected_figures/17...,png,6104051,5.821277
52,selected_figures/14_uncertainty_trustworthines...,outputs/supervisor_package/selected_figures/14...,png,5983064,5.705894
50,selected_figures/12_stable_diffusion_uncertain...,outputs/supervisor_package/selected_figures/12...,png,5126856,4.889351
56,selected_figures/18_uncertainty_trustworthines...,outputs/supervisor_package/selected_figures/18...,png,5126828,4.889324
49,selected_figures/11_stable_diffusion_uncertain...,outputs/supervisor_package/selected_figures/11...,png,5002172,4.770443
48,selected_figures/10_stable_diffusion_uncertain...,outputs/supervisor_package/selected_figures/10...,png,4796946,4.574724


In [23]:
required_supervisor_markdown_files = [
    supervisor_output_paths["readme"],
    supervisor_output_paths["methodology_summary"],
    supervisor_output_paths["results_summary"],
    supervisor_output_paths["proposal_alignment"],
    supervisor_output_paths["limitations_and_deviations"],
    supervisor_output_paths["supervisor_questions"],
    supervisor_output_paths["next_steps"],
]

required_supervisor_data_files = [
    supervisor_data_dir / "final_controlled_50_key_results_summary.csv",
    supervisor_data_dir / "final_controlled_50_dataset_summary.csv",
    supervisor_data_dir / "final_controlled_50_model_stack_summary.csv",
    supervisor_data_dir / "final_controlled_50_metric_policy_summary.csv",
    supervisor_data_dir / "final_controlled_50_model_win_summary.csv",
    supervisor_data_dir / "final_controlled_50_uncertainty_summary.csv",
    supervisor_data_dir / "final_controlled_50_sdxl_feasibility_summary.csv",
    supervisor_data_dir / "proposal_research_question_coverage.csv",
    supervisor_data_dir / "proposal_hypothesis_coverage.csv",
    supervisor_data_dir / "research_question_coverage_summary.csv",
    supervisor_data_dir / "supervisor_questions.csv",
    supervisor_data_dir / "selected_supervisor_visual_manifest.csv",
    supervisor_data_dir / "supervisor_report_file_sizes.csv",
    supervisor_data_dir / "package_manifest_files.csv",
]

required_supervisor_manifest_files = [
    supervisor_output_paths["package_manifest"],
]

required_supervisor_docs = [
    supervisor_docs_dir / "methodology_notes.md",
    supervisor_docs_dir / "literature_reference_log.md",
    supervisor_docs_dir / "model_audit_notes.md",
]

required_supervisor_reports = [
    supervisor_reports_dir / "final_controlled_50_evaluation_report.html",
    supervisor_reports_dir / "opencv_lama_stable_diffusion_refined_metric_comparison_report_50.html",
    supervisor_reports_dir / "stable_diffusion_uncertainty_report_50.html",
]

all_required_supervisor_files = (
    required_supervisor_markdown_files
    + required_supervisor_data_files
    + required_supervisor_manifest_files
    + required_supervisor_docs
    + required_supervisor_reports
)

missing_supervisor_files = [
    str(path)
    for path in all_required_supervisor_files
    if not Path(path).exists()
]

if missing_supervisor_files:
    raise FileNotFoundError(
        "Missing required supervisor package files:\n"
        + "\n".join(missing_supervisor_files)
    )

# Validate selected figures.
selected_visual_manifest_df = pd.read_csv(
    supervisor_data_dir / "selected_supervisor_visual_manifest.csv"
)

if len(selected_visual_manifest_df) < 10:
    raise ValueError(
        f"Expected at least 10 selected supervisor figures, found {len(selected_visual_manifest_df)}."
    )

missing_selected_figures = []

for figure_path in selected_visual_manifest_df["package_figure_path"]:
    resolved_figure_path = resolve_project_path(figure_path)

    if not resolved_figure_path.exists():
        missing_selected_figures.append(str(resolved_figure_path))

if missing_selected_figures:
    raise FileNotFoundError(
        "Missing copied supervisor figure files:\n"
        + "\n".join(missing_selected_figures)
    )

# Validate markdown contents.
readme_text = supervisor_output_paths["readme"].read_text(encoding="utf-8")
proposal_alignment_text = supervisor_output_paths["proposal_alignment"].read_text(encoding="utf-8")
questions_text = supervisor_output_paths["supervisor_questions"].read_text(encoding="utf-8")
results_text = supervisor_output_paths["results_summary"].read_text(encoding="utf-8")

required_text_phrases = [
    "Visual plausibility is not the same as restoration trustworthiness.",
    "evaluation framework",
    "50 paintings",
    "LaMa",
    "Stable Diffusion",
    "SDXL",
    "uncertainty",
]

combined_supervisor_text = "\n".join(
    [
        readme_text,
        proposal_alignment_text,
        questions_text,
        results_text,
    ]
)

missing_required_phrases = [
    phrase
    for phrase in required_text_phrases
    if phrase not in combined_supervisor_text
]

if missing_required_phrases:
    raise ValueError(
        "Supervisor package text missing required phrases:\n"
        + "\n".join(missing_required_phrases)
    )

# Validate package manifest.
saved_package_manifest = json.loads(
    supervisor_output_paths["package_manifest"].read_text(encoding="utf-8")
)

if len(saved_package_manifest["files"]) < 20:
    raise ValueError("Package manifest contains too few files.")

if saved_package_manifest["controlled_subset"]["paintings"] != 50:
    raise ValueError("Package manifest painting count mismatch.")

if saved_package_manifest["refined_comparison"]["lama_majority_cases"] != 155:
    raise ValueError("Package manifest LaMa majority-vote count mismatch.")

print("Supervisor package final gates passed.")
print("Package root:", supervisor_package_dir)
print("Required markdown files:", len(required_supervisor_markdown_files))
print("Required data files:", len(required_supervisor_data_files))
print("Required docs:", len(required_supervisor_docs))
print("Required reports:", len(required_supervisor_reports))
print("Selected copied figures:", len(selected_visual_manifest_df))
print("Package manifest files:", len(saved_package_manifest["files"]))

Supervisor package final gates passed.
Package root: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package
Required markdown files: 7
Required data files: 14
Required docs: 3
Required reports: 3
Selected copied figures: 18
Package manifest files: 58


In [24]:
package_size_summary_df = (
    package_manifest_df
    .assign(top_level_folder=lambda df: df["relative_path"].str.split("/").str[0])
    .groupby("top_level_folder", dropna=False)
    .agg(
        files=("relative_path", "count"),
        total_size_mb=("size_mb", "sum"),
        max_file_size_mb=("size_mb", "max"),
    )
    .reset_index()
    .sort_values("total_size_mb", ascending=False)
)

package_size_summary_path = supervisor_data_dir / "package_size_summary.csv"
package_size_summary_df.to_csv(package_size_summary_path, index=False)

display(package_size_summary_df)

large_package_files_df = package_manifest_df[
    package_manifest_df["size_mb"] > 50
].sort_values("size_mb", ascending=False)

large_package_files_path = supervisor_data_dir / "large_package_files_over_50mb.csv"
large_package_files_df.to_csv(large_package_files_path, index=False)

if not large_package_files_df.empty:
    print("Large package files over 50 MB detected:")
    display(large_package_files_df)
else:
    print("No package files over 50 MB.")

print("Saved package size summary:", package_size_summary_path)
print("Saved large-file summary:", large_package_files_path)

,top_level_folder,files,total_size_mb,max_file_size_mb
7,reports,3,403.547433,192.954260
9,selected_figures,18,69.152875,6.047616
2,docs,3,0.162577,0.073555
1,data,27,0.114332,0.065096
6,proposal_alignment.md,1,0.009240,0.009240
8,results_summary.md,1,0.007285,0.007285
4,methodology_summary.md,1,0.005095,0.005095
3,limitations_and_deviations.md,1,0.004844,0.004844
0,README_supervisor.md,1,0.003241,0.003241
10,supervisor_questions.md,1,0.003029,0.003029


Large package files over 50 MB detected:


,relative_path,project_relative_path,file_type,size_bytes,size_mb
35,reports/final_controlled_50_evaluation_report....,outputs/supervisor_package/reports/final_contr...,html,202327206,192.954260
37,reports/stable_diffusion_uncertainty_report_50...,outputs/supervisor_package/reports/stable_diff...,html,194945971,185.914966


Saved package size summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\data\package_size_summary.csv
Saved large-file summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\data\large_package_files_over_50mb.csv


## Notebook 30 summary

This notebook created a supervisor review package for the controlled 50-painting thesis evaluation.

The package was saved to:

`outputs/supervisor_package/`

The package includes:

- `README_supervisor.md`
- `proposal_alignment.md`
- `methodology_summary.md`
- `results_summary.md`
- `limitations_and_deviations.md`
- `supervisor_questions.md`
- `next_steps.md`
- `package_manifest.json`
- compact CSV/JSON summaries under `data/`
- selected visual figures under `selected_figures/`
- copied reports under `reports/`
- copied project notes under `docs/`

The package explicitly maps the current work back to the remembered thesis proposal framing:

- RQ1: multi-metric trustworthiness evaluation,
- RQ2: pretrained model comparison across painting categories and damage types,
- RQ3: diffusion uncertainty from multiple restoration candidates.

The current work substantially answers RQ1 and RQ2 for the controlled 50-painting subset.

RQ3 is answered diagnostically through the 40-case Stable Diffusion uncertainty subset, but supervisor confirmation is needed on whether this should be expanded to all 200 non-zero cases.

The package also identifies methodological questions for the supervisor:

- whether the 50-painting subset is sufficient,
- whether the 40-case uncertainty subset is sufficient,
- whether SDXL feasibility-only treatment is acceptable,
- whether the refined metric-region policy is acceptable,
- whether the dashboard should be formalized as a supporting artifact,
- whether final thesis work should scale dataset size, uncertainty analysis, or SDXL evaluation.

The package is intended for supervisor review before building the final Streamlit dashboard and thesis-writing assets.

In [25]:
def append_once(path: Path, marker: str, text: str) -> None:
    path = Path(path)

    if path.exists():
        existing_text = path.read_text(encoding="utf-8")
    else:
        existing_text = ""

    if marker in existing_text:
        print(f"Marker already exists in {path.name}; skipping append.")
        return

    updated_text = existing_text.rstrip() + "\n\n" + text.strip() + "\n"
    path.write_text(updated_text, encoding="utf-8")
    print(f"Updated {path}")


methodology_marker = "<!-- NOTEBOOK_30_SUPERVISOR_PACKAGE -->"

methodology_append_text = f"""
{methodology_marker}

## Supervisor package preparation

Notebook:

`notebooks/30_supervisor_package_cleaned.ipynb`

A compact supervisor review package was generated after the final controlled 50-painting report and dashboard asset preparation.

Package directory:

`outputs/supervisor_package/`

The package includes:

- current project status,
- proposal-to-current-work alignment,
- research-question coverage,
- key results,
- methodology summary,
- limitations and deviations,
- supervisor questions,
- recommended next steps,
- selected visual figures,
- compact CSV/JSON summaries,
- references/copies of main reports.

The package maps the current work back to the remembered proposal research questions:

- RQ1: multi-metric trustworthiness evaluation,
- RQ2: pretrained model comparison across painting categories and damage types,
- RQ3: diffusion uncertainty from multiple restoration candidates.

Current interpretation:

- RQ1 is substantially answered by the multi-metric, region-aware evaluation framework.
- RQ2 is substantially answered for the controlled 50-painting subset.
- RQ3 is diagnostically answered by the balanced 40-case Stable Diffusion uncertainty subset, but scope should be confirmed with the supervisor.

Main supervisor clarification points:

- whether the 50-painting subset is sufficient,
- whether uncertainty should be expanded to all 200 non-zero cases,
- whether SDXL should remain feasibility-audited or be rerun on university compute,
- whether the refined metric-region policy is accepted,
- whether the Streamlit dashboard should be formalized as a supporting artifact.
"""

model_audit_marker = "<!-- NOTEBOOK_30_SUPERVISOR_PACKAGE -->"

model_audit_append_text = f"""
{model_audit_marker}

### Supervisor package status

Notebook completed:

`notebooks/30_supervisor_package_cleaned.ipynb`

A supervisor-facing review package has been created at:

`outputs/supervisor_package/`

The package summarizes the current evaluated model stack:

- OpenCV Telea,
- LaMa,
- Stable Diffusion Inpainting,
- SDXL Inpainting as feasibility-audited only.

The package also documents the main model-related supervisor questions:

1. whether SDXL should remain feasibility-audited only,
2. whether university GPU resources should be requested for SDXL,
3. whether the 40-case Stable Diffusion uncertainty subset is sufficient,
4. whether the final thesis should emphasize the LaMa versus Stable Diffusion contrast as a central empirical result.

Current model interpretation remains:

- LaMa is strongest under the refined reference-based comparison.
- OpenCV Telea is useful as a deterministic baseline.
- Stable Diffusion exposes the gap between visual plausibility, reference-based performance, and generative stability.
- SDXL is excluded from full local comparison due feasibility, not as a model-quality conclusion.
"""

literature_marker = "<!-- NOTEBOOK_30_SUPERVISOR_PACKAGE -->"

literature_append_text = f"""
{literature_marker}

## 30. Supervisor package and proposal alignment

### Decision supported

A supervisor package was created to connect the completed controlled experiment back to the thesis proposal framing.

The package emphasizes that the thesis contribution is an evaluation framework rather than a new restoration model.

### Proposal alignment

The package maps the current experiment to three remembered research directions:

1. multi-metric restoration trustworthiness,
2. pretrained model comparison across painting and damage conditions,
3. diffusion uncertainty from multiple restoration candidates.

### Current interpretation

The completed 50-painting controlled experiment supports the main framework claim:

> Visual plausibility is not the same as restoration trustworthiness.

The package also identifies remaining clarification points for the supervisor, especially around final dataset scale, uncertainty subset size, SDXL feasibility, and dashboard formalization.

### Main artifact

`outputs/supervisor_package/`
"""

append_once(
    source_paths["methodology_notes"],
    methodology_marker,
    methodology_append_text,
)

append_once(
    source_paths["model_audit_notes"],
    model_audit_marker,
    model_audit_append_text,
)

append_once(
    source_paths["literature_reference_log"],
    literature_marker,
    literature_append_text,
)

Updated D:\Masters\FH\Thesis\painting-restoration-eval\docs\methodology_notes.md
Updated D:\Masters\FH\Thesis\painting-restoration-eval\docs\model_audit_notes.md
Updated D:\Masters\FH\Thesis\painting-restoration-eval\docs\literature_reference_log.md


In [26]:
important_git_files = [
    PROJECT_ROOT / "notebooks" / "30_supervisor_package_cleaned.ipynb",
    supervisor_output_paths["package_manifest"],
    supervisor_output_paths["readme"],
    supervisor_output_paths["proposal_alignment"],
    supervisor_output_paths["methodology_summary"],
    supervisor_output_paths["results_summary"],
    supervisor_output_paths["limitations_and_deviations"],
    supervisor_output_paths["supervisor_questions"],
    supervisor_output_paths["next_steps"],
    source_paths["methodology_notes"],
    source_paths["literature_reference_log"],
    source_paths["model_audit_notes"],
]

git_hygiene_rows = []

for path in important_git_files:
    if path.exists():
        git_hygiene_rows.append(
            {
                "path": to_project_relative_path(path),
                "size_mb": path.stat().st_size / (1024 * 1024),
                "exists": True,
                "large_warning_over_50mb": path.stat().st_size > 50 * 1024 * 1024,
                "hard_limit_risk_over_100mb": path.stat().st_size > 100 * 1024 * 1024,
            }
        )
    else:
        git_hygiene_rows.append(
            {
                "path": to_project_relative_path(path),
                "size_mb": np.nan,
                "exists": False,
                "large_warning_over_50mb": False,
                "hard_limit_risk_over_100mb": False,
            }
        )

git_hygiene_df = pd.DataFrame(git_hygiene_rows)

git_hygiene_path = supervisor_data_dir / "git_hygiene_summary_notebook_30.csv"
git_hygiene_df.to_csv(git_hygiene_path, index=False)

display(git_hygiene_df)

if git_hygiene_df["hard_limit_risk_over_100mb"].any():
    print("WARNING: One or more important files exceed 100 MB.")
elif git_hygiene_df["large_warning_over_50mb"].any():
    print("WARNING: One or more important files exceed 50 MB.")
else:
    print("No important Notebook 30 files exceed 50 MB.")

print("Saved Git hygiene summary:", git_hygiene_path)

,path,size_mb,exists,large_warning_over_50mb,hard_limit_risk_over_100mb
0,notebooks/30_supervisor_package_cleaned.ipynb,0.165346,True,False,False
1,outputs/supervisor_package/package_manifest.json,0.018031,True,False,False
2,outputs/supervisor_package/README_supervisor.md,0.003241,True,False,False
3,outputs/supervisor_package/proposal_alignment.md,0.009240,True,False,False
4,outputs/supervisor_package/methodology_summary.md,0.005095,True,False,False
5,outputs/supervisor_package/results_summary.md,0.007285,True,False,False
6,outputs/supervisor_package/limitations_and_dev...,0.004844,True,False,False
7,outputs/supervisor_package/supervisor_question...,0.003029,True,False,False
8,outputs/supervisor_package/next_steps.md,0.002502,True,False,False
9,docs/methodology_notes.md,0.056921,True,False,False


No important Notebook 30 files exceed 50 MB.
Saved Git hygiene summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\data\git_hygiene_summary_notebook_30.csv
